# AI Weekly Business Review — Walmart Store Sales
**Project 1 of 3 — GenAI Analytics Portfolio**

**Business problem:** Retail leadership reads a weekly business review every Monday. Analysts spend hours writing the commentary by hand: what happened, why, and whether it matters. The hard part isn't the numbers — it's judging which moves are normal (seasonality, holidays, regular swings) and which deserve attention.

**What this notebook does:**
1. Loads ~2.7 years of weekly sales for 45 Walmart stores (2010–2012)
2. Rolls it up to total chain sales per week
3. Builds **baselines** for every week: last week, the prior 13-week average and range, the same week last year, and the recent year-over-year trend — plus which key shopping days (e.g. Christmas Eve) fall in the week this year vs last year
4. Breaks each week down by store type, store and department
5. Sends that evidence to Claude, which writes a plain-English brief **every week** — and decides how much emphasis the week deserves, using the baselines as context
6. Saves a quarter of weekly briefs to a CSV, plus a second CSV with the detail behind each brief
7. Checks the output automatically (length and number accuracy) before you review it by hand

**Design choice:** there are no hard-coded anomaly thresholds. Earlier versions used a ±15% rule and then a z-score with status tiers; both produced labels that didn't match business sense on noisy data. Here Claude judges significance from multiple baselines, the way an analyst would — and Step 12 checks whether that judgment holds up.

**Before you run it:** add your Anthropic API key as a Colab Secret (key icon 🔑 in the left sidebar → *Add new secret* → name it `ANTHROPIC_API_KEY` → toggle *Notebook access* on).

## Step 1 — Install the Claude library
Colab already has pandas. We only need Anthropic's library. `-q` means "quiet".

In [ ]:
!pip install -q anthropic

## Step 2 — Load the data from Hugging Face
The dataset has two files we need:
- `train.csv` — weekly sales for every store × department (`Store`, `Dept`, `Date`, `Weekly_Sales`, `IsHoliday`)
- `stores.csv` — store type (A / B / C) and size in square feet

We read them straight from Hugging Face with `pd.read_csv`. We skip the dataset's `test.csv` because its sales column is blank (it was a Kaggle competition file).

The checks at the bottom are a habit worth keeping: confirm the data is what you think it is *before* building on it.

In [ ]:
import pandas as pd

BASE_URL = "https://huggingface.co/datasets/large-traversaal/Walmart-sales/resolve/main/"
sales = pd.read_csv(BASE_URL + "train.csv", parse_dates=["Date"])
stores = pd.read_csv(BASE_URL + "stores.csv")

sales = sales.merge(stores, on="Store", how="left")   # adds Type and Size to every row

print(f"Rows: {len(sales):,}")
print(f"Dates: {sales.Date.min().date()} to {sales.Date.max().date()}")
print(f"Stores: {sales.Store.nunique()}   Departments: {sales.Dept.nunique()}")
print(f"Missing sales values: {sales.Weekly_Sales.isna().sum()}")
print(f"Rows with negative sales (returns exceeding sales): {(sales.Weekly_Sales < 0).sum():,}")
sales.head()

## Step 3 — Roll up to total chain sales per week
- `groupby("Date")` adds up every store and department for each week — like a pivot table with Date as rows.
- `IsHoliday` is the same for every row in a week, so we take the first value.
- Each `Date` is the **Friday the week ends on** — each week runs Saturday to Friday. The dataset doesn't document this, so the cell checks it: under this reading, every flagged holiday falls inside its week (e.g. the 2010 Labor Day week is flagged on Fri 10 Sep, and Labor Day was Mon 6 Sep). Step 4's key-shopping-days logic depends on this, so the cell stops if the check fails.

**About the holiday flag:** Walmart flagged four holiday weeks a year — Super Bowl, Labor Day, Thanksgiving and Christmas. The flagged Christmas week is the week *containing* Christmas Day (mostly post-Christmas days), so the big pre-Christmas shopping week is **not** flagged. The system prompt tells Claude about this.

In [ ]:
weekly = (
    sales.groupby("Date")
         .agg(revenue=("Weekly_Sales", "sum"), is_holiday=("IsHoliday", "first"))
         .reset_index()
         .rename(columns={"Date": "week"})
         .sort_values("week")
         .reset_index(drop=True)
)

# Sanity check: weeks should be exactly 7 days apart with no gaps.
# The "same week last year" comparison in Step 4 depends on this.
gaps = weekly["week"].diff().dropna().dt.days
assert (gaps == 7).all(), "Weeks are not evenly spaced -- check the data"

# Check the date convention: is each Date the LAST day (Friday) of its week?
# If so, every flagged holiday should fall within the 7 days ending on its Date.
def holiday_date(week_end):
    y, mth = week_end.year, week_end.month
    if mth == 2:    # Super Bowl: first Sunday of February (true for 2010-2012)
        d = pd.Timestamp(y, 2, 1); return d + pd.Timedelta(days=(6 - d.weekday()) % 7)
    if mth == 9:    # Labor Day: first Monday of September
        d = pd.Timestamp(y, 9, 1); return d + pd.Timedelta(days=(0 - d.weekday()) % 7)
    if mth == 11:   # Thanksgiving: fourth Thursday of November
        d = pd.Timestamp(y, 11, 1); return d + pd.Timedelta(days=(3 - d.weekday()) % 7 + 21)
    if mth == 12:   # Christmas Day
        return pd.Timestamp(y, 12, 25)
    return None

flagged_weeks = weekly.loc[weekly["is_holiday"], "week"]
inside = [holiday_date(w) is not None and w - pd.Timedelta(days=6) <= holiday_date(w) <= w for w in flagged_weeks]
print(f"Date convention check: {sum(inside)} of {len(inside)} flagged holidays fall inside the week ENDING on their Date")
assert all(inside), "Date convention check failed -- Step 4's key-shopping-days logic would be wrong"

print(f"{len(weekly)} weeks, average ${weekly.revenue.mean() / 1e6:,.1f}M per week")
print("\nHoliday weeks:")
print(weekly[weekly.is_holiday][["week", "revenue"]].assign(revenue=lambda d: (d.revenue / 1e6).round(1)).to_string(index=False))

## Step 4 — Build the baselines for every week
No single comparison tells the whole story, so we give Claude several:

| Baseline | Question it answers | Catches |
|---|---|---|
| **vs last week** | Did something just change? | Sudden shifts |
| **vs prior 13-week average** (one quarter) | Is this normal for recent trading? | Trend breaks |
| **vs same week last year** | Is this normal *for this time of year*? | Seasonality and holidays |
| **vs the recent year-over-year trend** | Is this week's growth normal *for how the business has been trending*? | Stops a steady +5% year being treated as news every week |
| **key shopping days** this week vs the same week last year | Did a big shopping day land in a different week this year? | Calendar shifts — Christmas Eve fell in the week ending 24 Dec 2010, but in the week ending 30 Dec 2011 |

We also give Claude the **recent range** — the lowest and highest week in the prior 13 weeks — so it can say in plain English whether this week was inside or outside normal, and the **$ change** for each comparison, so it doesn't have to do subtraction itself (a common source of LLM arithmetic errors) and so Step 12 can check its numbers.

**Key shopping days:** only fixed-date days move between weeks — July 4, Halloween, Christmas Eve, Christmas Day and New Year's Eve. Holidays tied to a weekday (Thanksgiving, Super Bowl, Labor Day) always land in the same position, so they never shift. We also give Claude **next week's** key days. That's calendar information known in advance, not future sales, so it can warn leadership on Monday when a comparison is about to reverse.

`.shift(1)` means "use only the weeks *before* this one", so a week is never compared against itself. `.shift(52)` looks back 52 weeks — the same Friday-labelled week a year ago.

In [ ]:
BASELINE_WEEKS = 13

prior = weekly["revenue"].shift(1).rolling(BASELINE_WEEKS)          # the 13 weeks BEFORE each week
weekly["avg_13wk"] = prior.mean()
weekly["low_13wk"] = prior.min()
weekly["high_13wk"] = prior.max()
weekly["prev_week"] = weekly["revenue"].shift(1)
weekly["last_year"] = weekly["revenue"].shift(52)
weekly["last_year_holiday"] = weekly["is_holiday"].shift(52)
weekly["prev_week_holiday"] = weekly["is_holiday"].shift(1).fillna(False).astype(bool)
weekly["holiday_in_13wk"] = weekly["is_holiday"].astype(int).shift(1).rolling(BASELINE_WEEKS).max().fillna(0).astype(bool)

# % changes
weekly["wow_pct"] = weekly["revenue"].pct_change() * 100
weekly["vs_13wk_pct"] = (weekly["revenue"] / weekly["avg_13wk"] - 1) * 100
weekly["yoy_pct"] = (weekly["revenue"] / weekly["last_year"] - 1) * 100

# $ changes -- given to Claude so it doesn't have to subtract, and used in Step 12 to check its numbers
weekly["chg_wow"] = weekly["revenue"] - weekly["prev_week"]
weekly["chg_vs_13wk"] = weekly["revenue"] - weekly["avg_13wk"]
weekly["chg_vs_ly"] = weekly["revenue"] - weekly["last_year"]

# Recent year-over-year trend: average YoY growth over the 13 weeks BEFORE this one
weekly["yoy_trend_13wk"] = weekly["yoy_pct"].shift(1).rolling(BASELINE_WEEKS).mean()
weekly["vs_trend_pts"] = weekly["yoy_pct"] - weekly["yoy_trend_13wk"]   # percentage points above/below trend

# Key shopping days: fixed-date days that land in a different week each year
KEY_DAYS = {(7, 4): "July 4", (10, 31): "Halloween", (12, 24): "Christmas Eve",
            (12, 25): "Christmas Day", (12, 31): "New Year's Eve"}

def key_days_in_week(week_end):
    days = pd.date_range(week_end - pd.Timedelta(days=6), week_end)   # Saturday to Friday
    found = [KEY_DAYS[(d.month, d.day)] for d in days if (d.month, d.day) in KEY_DAYS]
    return ", ".join(found) if found else "none"

ONE_WEEK, ONE_YEAR = pd.Timedelta(weeks=1), pd.Timedelta(weeks=52)
weekly["key_days"] = weekly["week"].apply(key_days_in_week)
weekly["key_days_ly"] = (weekly["week"] - ONE_YEAR).apply(key_days_in_week)
weekly["key_days_next"] = (weekly["week"] + ONE_WEEK).apply(key_days_in_week)              # calendar only, no future sales
weekly["key_days_next_ly"] = (weekly["week"] + ONE_WEEK - ONE_YEAR).apply(key_days_in_week)

def position_in_range(row):
    if pd.isna(row["low_13wk"]):
        return "not enough history"
    if row["revenue"] > row["high_13wk"]:
        return "above the recent range"
    if row["revenue"] < row["low_13wk"]:
        return "below the recent range"
    return "within the recent range"

weekly["range_position"] = weekly.apply(position_in_range, axis=1)

# Weeks with every baseline available (the YoY trend needs a year plus a quarter of history)
reportable = weekly.dropna(subset=["avg_13wk", "last_year", "yoy_trend_13wk"]).reset_index(drop=True)
print(f"{len(reportable)} weeks have all baselines "
      f"({reportable.week.min().date()} to {reportable.week.max().date()})")

print("\nKey shopping days in late-December weeks, this year vs same week last year:")
dec = weekly[(weekly["week"].dt.month == 12) & (weekly["week"].dt.day >= 17)]
print(dec[["week", "key_days", "key_days_ly"]].to_string(index=False), "\n")

reportable[["week", "revenue", "wow_pct", "vs_13wk_pct", "yoy_pct", "yoy_trend_13wk", "vs_trend_pts", "range_position", "is_holiday"]].tail(8).round(
    {"revenue": 0, "wow_pct": 1, "vs_13wk_pct": 1, "yoy_pct": 1, "yoy_trend_13wk": 1, "vs_trend_pts": 1})

### Quick visual check
The solid line is this year; the dashed line is the same week last year. Where the two lines move together, the swing is **seasonal** — expected. Where they split apart, something genuinely changed. Red dots mark Walmart's flagged holiday weeks.

This chart is a good one to screenshot for the README.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(weekly["week"], weekly["revenue"] / 1e6, linewidth=1.2, label="Weekly sales")
ax.plot(weekly["week"], weekly["last_year"] / 1e6, linewidth=1, linestyle="--", label="Same week last year")
holidays = weekly[weekly["is_holiday"]]
ax.scatter(holidays["week"], holidays["revenue"] / 1e6, color="red", zorder=3, label="Flagged holiday week")
ax.set_ylabel("Sales ($M)")
ax.set_title("Walmart weekly sales vs same week last year")
ax.legend()
plt.show()

## Step 5 — Gather the evidence behind each week
**This is the most important step.** If you only tell Claude "sales were up 12%", it will *guess* at reasons. Instead we hand it the actual breakdown, compared against both the 13-week average and last year:
- **By store type** (A = large supercentres, B = mid-size, C = small)
- **Top 5 stores** that moved most
- **Top 5 departments** that moved most

Each table also shows `pct_vs_ly` (% change vs last year) and `share_of_sales_%`. These stop a common misreading: large stores always contribute the most *dollars* because they're the biggest, so a change is only "concentrated" if one group's *percentage* change is out of line with the others.

Departments are numbered (e.g. Dept 72) with no names in this dataset, so the prompt tells Claude not to guess what they sell.

In [ ]:
def get_week_drivers(week):
    """Compare one week's mix to the prior 13-week average and the same week last year."""
    base_start = week - pd.Timedelta(weeks=BASELINE_WEEKS)
    ly_week = week - pd.Timedelta(weeks=52)

    this_wk = sales[sales["Date"] == week]
    base = sales[(sales["Date"] >= base_start) & (sales["Date"] < week)]
    ly = sales[sales["Date"] == ly_week]

    def compare(dim, top_n=None):
        out = pd.DataFrame({
            "this_week": this_wk.groupby(dim)["Weekly_Sales"].sum(),
            "avg_13wk": base.groupby(dim)["Weekly_Sales"].sum() / BASELINE_WEEKS,
            "last_year": ly.groupby(dim)["Weekly_Sales"].sum(),
        }).fillna(0)
        out["chg_vs_13wk"] = out["this_week"] - out["avg_13wk"]
        out["chg_vs_ly"] = out["this_week"] - out["last_year"]
        # Percentages are kept separately so they aren't divided by 1,000 below
        pct_vs_ly = (out["chg_vs_ly"] / out["last_year"].where(out["last_year"] != 0) * 100).round(1)
        share = (out["this_week"] / out["this_week"].sum() * 100).round(1)
        out = (out / 1e3).round(0)   # $ columns in thousands, easier to read
        out["pct_vs_ly"] = pct_vs_ly
        out["share_of_sales_%"] = share
        out = out.sort_values("chg_vs_ly", key=abs, ascending=False)   # biggest movers vs last year first
        if top_n:
            out = out.head(top_n)
        return out

    return {
        "by_store_type": compare("Type"),
        "top_stores": compare("Store", top_n=5),
        "top_departments": compare("Dept", top_n=5),
        "stores_up_vs_ly": int((this_wk.groupby("Store")["Weekly_Sales"].sum()
                                > ly.groupby("Store")["Weekly_Sales"].sum()).sum()),
        "store_count": this_wk["Store"].nunique(),
    }

example = get_week_drivers(reportable.iloc[-1]["week"])
print("$ figures in thousands; pct_vs_ly and share_of_sales_% are percentages\n")
print(example["by_store_type"], "\n")
print(example["top_departments"])
print(f"\nStores up vs last year: {example['stores_up_vs_ly']} of {example['store_count']}")

## Step 6 — Connect to Claude and write the system prompt
- `userdata.get(...)` reads your key from Colab Secrets so it never appears in the notebook (critical before pushing to GitHub).
- The **system prompt** does the job the thresholds used to do. It tells Claude *how to judge* significance:
  - Last year is the most important comparison, because it controls for seasonality.
  - A week that's in its normal range and close to last year is routine — say so briefly and don't invent concern.
  - Only use the numbers provided; never invent causes.
- We ask for **JSON** so each part lands in its own CSV column.
- **Order matters.** `attention_level` comes *first* in the JSON, so Claude decides how much the week matters before it writes. If the judgment came last, it would write the full summary before deciding the week was routine.
- **Length limits are stated per level and override the other rules**, plus one worked example of a routine week. Without these, rules like "explain the drivers" made every week long.

In [ ]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-5-5"   # fast and cheap for looping; swap for a larger model to compare quality

SYSTEM_PROMPT = """You are a senior retail analyst writing the weekly business review for Walmart's US leadership team (merchandising, operations, and finance). Leadership reads this every Monday and values brevity.

STEP 1 - Decide the attention level BEFORE writing anything else.
Judge year-over-year growth against the business's RECENT YEAR-OVER-YEAR TREND, not against zero. If the business has been running +5% versus last year, a +5% week is normal; a -1% week is a 6-point slowdown.
- "routine": year-over-year growth is within about 2 points of the recent trend, or the gap is explained by a key shopping day falling in a different week than last year, or the swing is explained by a holiday this week or last week.
- "notable": year-over-year growth is roughly 2-5 points away from trend and not explained by holidays or calendar shifts, or one store type or department has a percentage change far out of line with the rest.
- "needs attention": more than about 5 points away from trend with no holiday or calendar explanation, or a broad decline across most stores.

STEP 2 - Write to the length the level allows. These limits override every other rule:
- routine: summary is ONE sentence, max 30 words. key_drivers is at most one short sentence naming the single largest offsetting department or store move, or "No material drivers". Do not discuss store types or how many stores were up unless it is the headline.
- notable: summary is 2-3 sentences. key_drivers names the 1-2 biggest drivers with $ amounts.
- needs attention: summary is 3-4 sentences. key_drivers names the 1-2 biggest drivers with $ amounts and says whether the move is broad-based or concentrated.

Rules:
- Use ONLY the numbers provided. Never invent causes such as weather, competitors, promotions, or the economy.
- Holiday flags are real data: Walmart flagged Super Bowl, Labor Day, Thanksgiving, and Christmas weeks. You are told whether THIS week and LAST week were flagged. A big drop the week after a holiday is the holiday unwinding - say so in a few words; it is not news. The flagged Christmas week is the week CONTAINING Christmas Day (mostly post-Christmas days); the pre-Christmas shopping week is not flagged. If a December week before Dec 25 spikes, you may say "likely pre-Christmas shopping" as a hypothesis.
- If the 13-week window contains a holiday week, its average and range are inflated; lean on the last-year comparison instead.
- Key shopping days: you are told which fixed-date shopping days (July 4, Halloween, Christmas Eve, Christmas Day, New Year's Eve) fall in this week, in the same week last year, and in next week. If this week and last year differ, say the year-over-year comparison is likely distorted by calendar timing and name the day. Do not call a calendar-driven gap a decline or a surge. If next week's key days also differ from last year, say which way next week's comparison is likely to move. This is a likely explanation, not a measured one - never estimate how many dollars a day is worth.
- You are given the $ change for each comparison. Use those figures rather than calculating your own.
- Judge broad-based versus concentrated using percentage changes and store counts, never dollar size. Large (Type A) stores are most of total sales, so they always contribute the most dollars. A move is concentrated only if one group's % change is far out of line with the others.
- Departments are identified by number only. Write "Dept 72" etc. and never guess what they sell.
- Lead with dollars and percentages, not statistics jargon.

Example of a correct routine week (numbers are illustrative only - never reuse them):
{"attention_level": "routine", "attention_reason": "Year-over-year growth of +2.1% is in line with the recent +2.4% trend", "headline": "Sales of $52.0M, up 2.1% on last year and in line with trend.", "summary": "A normal week: growth matched the recent year-over-year trend and sales stayed inside the recent range.", "key_drivers": "Dept 10 was the largest drag, down $0.3M (-9%) vs last year.", "watch_next_week": "Whether Dept 10 stays below last year."}

Return ONLY valid JSON with exactly these keys, in this order:
{
  "attention_level": "routine, notable, or needs attention",
  "attention_reason": "one short phrase citing which baseline or holiday drove the level",
  "headline": "one sentence, max 20 words, what happened in dollars",
  "summary": "length set by the attention level as above",
  "key_drivers": "as set by the attention level above",
  "watch_next_week": "exactly ONE thing to monitor next week, max 20 words, or 'Nothing specific'"
}"""
print("Client ready.")

## Step 7 — Build the prompt for one week and call Claude
`build_user_prompt` turns one week's numbers into a message. `write_weekly_brief` sends it and turns the JSON reply into a Python dictionary.

**`DRY_RUN = True` (the default) means no API calls and no cost.** Instead of calling Claude, the notebook shows you the exact prompt it *would* send and returns a placeholder brief. Check the numbers in the prompts first, then set `DRY_RUN = False` and re-run from this cell to generate real briefs.

Claude's reply can arrive in several blocks — sometimes a *thinking* block (the model reasoning first) before the answer — so we keep only the text blocks. Two safeguards keep the loop from producing broken rows:
- **`extract_json`** finds the JSON object even if Claude adds words before or after it.
- **A retry:** if a reply still can't be read, the notebook asks once more. If that fails too, the row is marked `PARSE_ERROR` and the `parse_status` column records why: `stop_reason=max_tokens` means the reply was cut off, and the first part of any text it did return is saved.

In [ ]:
import json

DRY_RUN = True   # True = no API calls, no cost. Set to False once the prompts look right.

def m(x):
    """Format dollars as $M with one decimal."""
    return f"${x / 1e6:,.1f}M"

def sm(x):
    """Format a $ change with its sign, e.g. +$2.3M or -$0.5M."""
    return f"{'+' if x >= 0 else '-'}${abs(x) / 1e6:,.1f}M"

def build_user_prompt(row, d):
    ly_holiday_note = ""
    if row["is_holiday"] != row["last_year_holiday"]:
        ly_holiday_note = (f"\nNOTE: holiday flag this week = {row['is_holiday']}, "
                           f"same week last year = {row['last_year_holiday']}. The holiday timing differs.")

    week_start = row["week"] - pd.Timedelta(days=6)
    return f"""Week ending: {row['week'].strftime('%A %d %B %Y')} (covers {week_start.strftime('%a %d %b')} to {row['week'].strftime('%a %d %b')})
Holiday week (Walmart flag): {row['is_holiday']}{ly_holiday_note}
Last week was a holiday week: {row['prev_week_holiday']}
Prior 13 weeks include a holiday week: {row['holiday_in_13wk']}
Key shopping days this week: {row['key_days']}
Key shopping days in the same week last year: {row['key_days_ly']}
Key shopping days next week: {row['key_days_next']} (same week last year: {row['key_days_next_ly']})

Total sales this week: {m(row['revenue'])}
vs last week ({m(row['prev_week'])}): {row['wow_pct']:+.1f}% ({sm(row['chg_wow'])})
vs prior 13-week average ({m(row['avg_13wk'])}): {row['vs_13wk_pct']:+.1f}% ({sm(row['chg_vs_13wk'])})
Prior 13-week range: {m(row['low_13wk'])} to {m(row['high_13wk'])} -> this week is {row['range_position']}
vs same week last year ({m(row['last_year'])}): {row['yoy_pct']:+.1f}% ({sm(row['chg_vs_ly'])})
Recent year-over-year trend (average of prior 13 weeks): {row['yoy_trend_13wk']:+.1f}% -> this week is {row['vs_trend_pts']:+.1f} points vs trend
Stores up vs last year: {d['stores_up_vs_ly']} of {d['store_count']}

In the tables below, $ columns are in thousands; pct_vs_ly and share_of_sales_% are percentages.

By store type (A = large, B = mid-size, C = small):
{d['by_store_type'].to_string()}

Top 5 stores by change vs last year:
{d['top_stores'].to_string()}

Top 5 departments by change vs last year:
{d['top_departments'].to_string()}

Write this week's business review."""


def extract_json(text):
    """Pull the JSON object out of a reply, ignoring any words or code fences around it."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found")
    return json.loads(text[start:end + 1])


def write_weekly_brief(row, max_attempts=2):
    drivers = get_week_drivers(row["week"])
    prompt = build_user_prompt(row, drivers)

    if DRY_RUN:
        # No API call: return a placeholder so the rest of the notebook still runs
        return {"headline": "DRY RUN - no API call made", "summary": f"Prompt is ~{len(prompt) // 4} tokens",
                "key_drivers": None, "attention_level": None, "attention_reason": None,
                "watch_next_week": None, "parse_status": "dry run"}

    for attempt in range(1, max_attempts + 1):
        response = client.messages.create(
            model=MODEL,
            max_tokens=4000,   # thinking counts toward this limit, so leave plenty of room for the answer
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": prompt}],
        )
        text = "".join(block.text for block in response.content if block.type == "text").strip()
        try:
            brief = extract_json(text)
            brief["parse_status"] = "ok" if attempt == 1 else f"ok after retry {attempt}"
            return brief
        except (ValueError, json.JSONDecodeError):
            print(f"   ! attempt {attempt}: could not parse reply (stop_reason={response.stop_reason})")

    # Still failing after retries: keep the raw reply so you can see what went wrong
    return {"headline": "PARSE_ERROR", "summary": None, "key_drivers": None,
            "attention_level": None, "attention_reason": None, "watch_next_week": None,
            "parse_status": f"failed: stop_reason={response.stop_reason}; reply={text[:300] or '(no text)'}"}

# Preview exactly what Claude receives for the most recent week
print(build_user_prompt(reportable.iloc[-1], example))

## Step 8 — Test on THREE contrasting weeks before looping
Each call costs a fraction of a cent, so test before the full loop. We pick:
- a **quiet week**: inside its recent range and close to last year
- **Thanksgiving week 2011**: a huge spike vs recent weeks, but expected for the season
- **Week ending 23 Dec 2011**: down vs last year partly because last year's week contained Christmas Eve and this year's doesn't

In dry-run mode this cell only prints the two prompts, which is the cheapest validation there is: if the numbers or holiday flags in the prompt are wrong, the brief will be wrong too. Once real briefs come back, the real test is calibration. The quiet week should be short and calm. Thanksgiving should be explained as seasonal (compared with last year's Thanksgiving), **not** treated as alarming just because it's far above the 13-week average. The 23 Dec week should name the Christmas Eve shift and say next week's comparison will look better, **not** call it a decline.

In [ ]:
quiet_pool = reportable[(reportable["range_position"] == "within the recent range") & (~reportable["is_holiday"])]
quiet_week = quiet_pool.loc[quiet_pool["yoy_pct"].abs().idxmin()]          # closest to last year
thanksgiving = reportable[reportable["week"] == "2011-11-25"].iloc[0]
pre_christmas = reportable[reportable["week"] == "2011-12-23"].iloc[0]

for row in [quiet_week, thanksgiving, pre_christmas]:
    print(f"=== {row['week'].date()} | {row['yoy_pct']:+.1f}% vs LY | {row['vs_13wk_pct']:+.1f}% vs 13-wk avg ===")
    if DRY_RUN:
        print(build_user_prompt(row, get_week_drivers(row["week"])), "\n")   # what Claude would see
    print(json.dumps(write_weekly_brief(row), indent=2), "\n")

## Step 9 — Write a brief for every week in the report period
In real life this runs once every Monday. For the portfolio we generate **one quarter (13 weeks)**.

**Development vs evaluation — avoid grading on the data you tuned on.**
- **Development quarter: Oct–Dec 2011.** The prompt was refined by reading these briefs, so results here will look better than they really are.
- **Evaluation quarter: Jul–Sep 2012.** Never used to write the prompt. Use *this* run as the evidence in your project report. Switch by changing `REPORT_END` below.

`time.sleep(1)` adds a short pause so we don't hit rate limits.

In [ ]:
import time

REPORT_WEEKS = 13
REPORT_END = pd.Timestamp("2011-12-30")   # development quarter (Oct-Dec 2011)
# REPORT_END = pd.Timestamp("2012-09-28") # evaluation quarter (Jul-Sep 2012) -- use for the final report

report = (reportable[reportable["week"] <= REPORT_END]
          .tail(REPORT_WEEKS)
          .reset_index(drop=True))

results = []
for i, row in report.iterrows():
    print(f"[{i + 1}/{len(report)}] {row['week'].date()}  "
          f"{row['vs_13wk_pct']:+6.1f}% vs 13wk  {row['yoy_pct']:+6.1f}% vs LY  "
          f"{'HOLIDAY' if row['is_holiday'] else ''}")
    results.append(write_weekly_brief(row))
    time.sleep(1)

print("Done.")

## Step 10 — Save the output: the briefs, plus the numbers to check them against
Two files:
- **`weekly_business_review.csv`** — one row per week: the comparison numbers on the left (including last year's sales, the 13-week average and the range, in $M), Claude's brief on the right. Every headline figure can be checked by reading across the row.
- **`weekly_drivers.csv`** — the store-type, store and department tables Claude saw, one row per item. Filter to a week in Excel to check any driver claim (e.g. "Dept 72 fell $526K").

The first file is your **sample output** for GitHub; the second is the audit trail behind it.

In [ ]:
briefs = pd.DataFrame(results)

# File 1: the comparison numbers behind every brief
numbers = pd.DataFrame({
    "week_ending": report["week"].dt.date,
    "is_holiday": report["is_holiday"],
    "key_days": report["key_days"],
    "key_days_ly": report["key_days_ly"],
    "revenue_$M": report["revenue"] / 1e6,
    "prev_week_$M": report["prev_week"] / 1e6,
    "avg_13wk_$M": report["avg_13wk"] / 1e6,
    "range_low_$M": report["low_13wk"] / 1e6,
    "range_high_$M": report["high_13wk"] / 1e6,
    "last_year_$M": report["last_year"] / 1e6,
    "chg_vs_ly_$M": report["chg_vs_ly"] / 1e6,
    "wow_pct": report["wow_pct"],
    "vs_13wk_pct": report["vs_13wk_pct"],
    "yoy_pct": report["yoy_pct"],
    "yoy_trend_13wk": report["yoy_trend_13wk"],
    "vs_trend_pts": report["vs_trend_pts"],
    "range_position": report["range_position"],
}).round(2)

# File 2: the store-type / store / department detail Claude saw, one row per item
week_drivers = {w: get_week_drivers(w) for w in report["week"]}
tables = []
for w, d in week_drivers.items():
    for label, key, prefix in [("Store type", "by_store_type", "Type "), ("Store", "top_stores", "Store "),
                               ("Department", "top_departments", "Dept ")]:
        t = d[key].reset_index()
        t = t.rename(columns={t.columns[0]: "item"})
        t["item"] = prefix + t["item"].astype(str)
        t.insert(0, "breakdown", label)
        t.insert(0, "week_ending", w.date())
        tables.append(t)
drivers_long = pd.concat(tables, ignore_index=True).rename(columns={
    "this_week": "this_week_$K", "avg_13wk": "avg_13wk_$K", "last_year": "last_year_$K",
    "chg_vs_13wk": "chg_vs_13wk_$K", "chg_vs_ly": "chg_vs_ly_$K"})

numbers["stores_up_vs_ly"] = [week_drivers[w]["stores_up_vs_ly"] for w in report["week"]]
numbers["store_count"] = [week_drivers[w]["store_count"] for w in report["week"]]

output = pd.concat([numbers, briefs], axis=1)
output.to_csv("weekly_business_review.csv", index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows dashes correctly
drivers_long.to_csv("weekly_drivers.csv", index=False, encoding="utf-8-sig")

from google.colab import files
files.download("weekly_business_review.csv")
files.download("weekly_drivers.csv")

print(output["parse_status"].value_counts().to_string(), "\n")   # every row should be "ok"
output[["week_ending", "yoy_pct", "vs_trend_pts", "key_days", "key_days_ly", "attention_level", "headline"]]

## Step 11 — Preview the latest brief the way leadership would see it
This formats one week as the text that would go into a Monday email or Slack post. Later (Phase B in your roadmap) this is the piece that gets sent automatically.

In [ ]:
icons = {"routine": "🟢", "notable": "🟡", "needs attention": "🔴"}
last = output.iloc[-1]

print(f"{icons.get(last['attention_level'], '')} WEEKLY BUSINESS REVIEW -- week ending {last['week_ending']}")
print(f"Sales: ${last['revenue_$M']:,.1f}M  |  {last['yoy_pct']:+.1f}% vs last year  |  "
      f"{last['vs_13wk_pct']:+.1f}% vs 13-week average\n")
print(last["headline"], "\n")
print(last["summary"], "\n")
print(f"Drivers: {last['key_drivers']}")
print(f"Watch next week: {last['watch_next_week']}")

## Step 12 — Validate the output (do this before publishing anything)
Three layers, from fully automatic to fully manual. **Write down what you find** — it becomes the evaluation section of your project report.

**12a. Length check (automatic).** Counts sentences in each summary and checks it against the limit for its attention level (`length_ok`).

**12b. Grounding check (automatic).** Pulls every $ figure, % figure and "N of 45 stores" count out of each brief and checks it matches a number Claude was given for that week, allowing for rounding. `unmatched_figures` lists the ones it couldn't find.
- An unmatched figure is **not automatically wrong.** Claude sometimes adds numbers together ("their combined $880K") or works out a share ("80% of the gain"). Check each one against `weekly_drivers.csv`.
- A matched figure is **not automatically right.** The check confirms the number exists, not that it's attached to the right store or department, or the right direction (up vs down).
- It only compares Claude against the numbers it was given. If the pipeline itself calculated something wrong, both would agree — so spot-check a couple of source numbers against the raw data as well.

**12c. Manual review (you).**
1. **Spot-check 2–3 weeks fully:** filter `weekly_drivers.csv` to the week and check every figure is attached to the right item and direction. Then check every `unmatched_figures` entry. This also tells you whether the grounding check can be trusted.
2. **No invented causes:** does any brief mention weather, promotions, competitors or the economy?
3. **Seasonality and calendar:** Thanksgiving explained as seasonal? Do weeks where `key_days` differs from `key_days_ly` (e.g. 23 Dec and 30 Dec 2011) mention calendar timing rather than calling it a decline or surge?
4. **Calibration:** do weeks with similar numbers get the same `attention_level`? Is the week after Thanksgiving routine?
5. **Departments:** does Claude ever guess what a numbered department sells?
6. **Usefulness:** would a merchandising leader learn something from each brief, or is it restating the numbers?

In [ ]:
import re

# ---------- 12a: length check ----------
LIMITS = {"routine": 1, "notable": 3, "needs attention": 4}   # max sentences allowed

def count_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(re.findall(r"[.!?](?:\s|$)", text.strip()))

output["summary_sentences"] = output["summary"].apply(count_sentences)
output["length_ok"] = output.apply(
    lambda r: r["attention_level"] in LIMITS and r["summary_sentences"] <= LIMITS[r["attention_level"]], axis=1)

# ---------- 12b: grounding check ----------
TEXT_FIELDS = ["headline", "summary", "key_drivers", "attention_reason", "watch_next_week"]
SCALE = {"m": 1e6, "million": 1e6, "k": 1e3, "thousand": 1e3, "b": 1e9, "billion": 1e9, "": 1}
DOLLAR_RE = re.compile(r"\$\s?(\d[\d,]*(?:\.\d+)?)\s?(million|thousand|billion|[mkb])?\b", re.I)
PCT_RE = re.compile(r"(\d+(?:\.\d+)?)\s?(?:%|percent|percentage points?|points?|pts)", re.I)
COUNT_RE = re.compile(r"\b(\d+)\s+of\s+\d+\s+stores", re.I)

def source_numbers(row, d):
    # Every number Claude was given for this week: dollars, percentages and store counts
    dollars = [row[c] for c in ["revenue", "prev_week", "avg_13wk", "low_13wk", "high_13wk", "last_year",
                                "chg_wow", "chg_vs_13wk", "chg_vs_ly"]]
    pcts = [row[c] for c in ["wow_pct", "vs_13wk_pct", "yoy_pct", "yoy_trend_13wk", "vs_trend_pts"]]
    for key in ["by_store_type", "top_stores", "top_departments"]:
        t = d[key]
        for col in ["this_week", "avg_13wk", "last_year", "chg_vs_13wk", "chg_vs_ly"]:
            dollars += list(t[col] * 1e3)                      # tables are in $ thousands
        pcts += list(t["pct_vs_ly"].dropna()) + list(t["share_of_sales_%"])
    counts = [d["stores_up_vs_ly"], d["store_count"] - d["stores_up_vs_ly"], d["store_count"]]
    return [x for x in dollars if pd.notna(x)], [x for x in pcts if pd.notna(x)], counts

def decimals(num):
    return len(num.split(".")[1]) if "." in num else 0

def check_brief(brief, row, d):
    text = " ".join(str(brief.get(f) or "") for f in TEXT_FIELDS)
    dollars, pcts, counts = source_numbers(row, d)
    cited, unmatched = 0, []
    for num, unit in DOLLAR_RE.findall(text):
        scale = SCALE[unit.lower()]
        value = float(num.replace(",", "")) * scale
        tol = 0.5 * 10 ** -decimals(num) * scale + 600 + 0.002 * value   # rounding in the brief and in the source
        cited += 1
        if not any(abs(value - abs(x)) <= tol for x in dollars):        # direction (up/down) is checked by hand
            unmatched.append(f"${num}{unit}")
    for num in PCT_RE.findall(text):
        value = float(num)
        tol = 0.5 * 10 ** -decimals(num) + 0.06
        cited += 1
        if not any(abs(value - abs(x)) <= tol for x in pcts):
            unmatched.append(f"{num}%")
    for num in COUNT_RE.findall(text):
        cited += 1
        if int(num) not in counts:
            unmatched.append(f"{num} stores")
    return cited, unmatched

checks = [check_brief(results[i], row, week_drivers[row["week"]]) for i, row in report.iterrows()]
output["figures_cited"] = [c for c, _ in checks]
output["figures_matched"] = [c - len(u) for c, u in checks]
output["unmatched_figures"] = ["; ".join(u) for _, u in checks]

# ---------- summary ----------
total_cited, total_matched = output["figures_cited"].sum(), output["figures_matched"].sum()
print("Attention levels:", output["attention_level"].value_counts().to_dict())
print(f"Length:    {output.length_ok.sum()} of {len(output)} summaries within their limit")
if total_cited:
    print(f"Grounding: {total_matched} of {total_cited} figures matched source data ({total_matched / total_cited:.0%})")
    print(f"           {(output.unmatched_figures != '').sum()} weeks have figures to check by hand\n")
else:
    print("Grounding: no figures to check (dry run?)\n")

# Re-save the main file with the validation columns added
output.to_csv("weekly_business_review.csv", index=False, encoding="utf-8-sig")
files.download("weekly_business_review.csv")

pd.set_option("display.max_colwidth", 200)
output[["week_ending", "yoy_pct", "vs_trend_pts", "key_days", "key_days_ly", "attention_level",
        "length_ok", "figures_cited", "figures_matched", "unmatched_figures", "summary"]]

## What to do after this runs
1. **Complete Step 12** and note the results — the grounding rate (e.g. "47 of 50 figures matched") and anything the manual review caught. That's your evidence and your "it failed when…" story for the README and LinkedIn post.
2. **Record what you changed and why:** ±15% rule → z-score tiers → dropped thresholds for multi-baseline judgment; Superstore → Walmart because Superstore was too sparse at weekly level.
3. **Push to GitHub:** this notebook + `weekly_business_review.csv` + `weekly_drivers.csv` + the chart screenshot + a README.
4. **Known limitation / next improvement:** each week is written in isolation, so the brief can't say "Dept 72 has been up 40% for four weeks". Passing the previous weeks' results into the prompt would fix this.
5. **Next (Project 5):** add a Prophet forecast so the review also says what's *coming*.